<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c21-header.png" alt="Nextia Learning · Scaling APIs and AI Workloads" width="100%">

# Capacity plan from a public traffic trace

**[Capacity plan from a public traffic trace](https://learning.nextia-ai.com/courses/scaling/m07/capacity-plan-from-a-public-traffic-trace/)** · Scaling APIs and AI Workloads · Module 7, case study 1 of 2 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** turn a real public request log into a capacity plan: check the trace, measure arrivals at three time scales, find the bursts, replay them through a queue with the course's measured AI job times, choose a number of job slots for a target wait, test the choice once on held-out months, and see what an outage and a queue bound would do.

| | |
|---|---|
| **Time** | About 2 hours with the case-study page. The notebook runs in about 2 minutes. |
| **Needs** | An internet connection for the setup cells. No account, no key, no money. The first cell installs pandas 3.0.6 and matplotlib. |
| **You should know** | The whole course, most of all [Workload model](https://learning.nextia-ai.com/courses/scaling/m01/workload-model/), [Latency and capacity](https://learning.nextia-ai.com/courses/scaling/m01/latency-and-capacity/), [Backpressure](https://learning.nextia-ai.com/courses/scaling/m04/backpressure/), [Scaling signals](https://learning.nextia-ai.com/courses/scaling/m05/scaling-signals/) and [Capacity recommendation](https://learning.nextia-ai.com/courses/scaling/m06/capacity-recommendation/). |
| **Data** | San Francisco 311 Cases (DataSF), 2025 and June 2026: the time, channel and type of each request. ODC Public Domain Dedication and License 1.0. The course's recorded AI job times (CC0). |
| **Tested** | Python 3.12.13 with pandas 3.0.6, numpy 2.5.3 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/scaling/m07/capacity-plan-from-a-public-traffic-trace/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C21/M07-L01-capacity-plan-from-a-public-traffic-trace/capacity-plan-from-a-public-traffic-trace-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6, the version that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see another version, restart the kernel (in Colab: **Runtime › Restart session**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 matplotlib
import pandas; print("pandas", pandas.__version__)

## Setup: get the data

Run the next cell. It downloads three files from the Nextia labs repository into a folder `capacity-plan`, and checks each file's checksum (SHA-256):

- `sf311_2025_arrivals.csv.gz` (about 4 MB): every San Francisco 311 request of 2025, with its time, channel and type;
- `sf311_2026_06_arrivals.csv.gz` (about 0.3 MB): the same for June 2026, the new input at the end;
- `calibration.json`: the course's recorded times of the three AI tasks.

You should see `Ready:` and the three files. If a copy of a file is next to this notebook, the cell uses that copy.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `capacity-plan` (in Colab: the **Files** panel on the left), then run the cell again.

Data: "311 Cases", City and County of San Francisco, DataSF (data.sfgov.org, dataset vw6y-z8j6), ODC Public Domain Dedication and License 1.0. Nextia Learning kept three columns (time, channel, type), removed the request ID and the always-zero milliseconds, and froze a copy, because the live dataset changes every day.

In [ ]:
# Setup: download the data files into capacity-plan/ and check them. Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/"
FILES = {
    "sf311_2025_arrivals.csv.gz": ("C21/M07-L01-capacity-plan-from-a-public-traffic-trace", "57c4209ce8aefb694a99436faef07d3eb6256e1004724fc5ec358ccdc720e548"),
    "sf311_2026_06_arrivals.csv.gz": ("C21/M07-L01-capacity-plan-from-a-public-traffic-trace", "3ea66ddeb5f9ad1cf412dce8253410a0b98958f49337941e305fb30851c599fc"),
    "calibration.json": ("C21/data", "ca7ee9f5c92a139443e4351213eabff9330fe45287c642fa8e8ca7286f1db8b5"),
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
HERE, WORK = Path.cwd(), Path("capacity-plan").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for name, (folder, expected) in FILES.items():
    target = WORK / name
    if not (target.exists() and sha256(target) == expected):
        if (HERE / name).exists():
            shutil.copy(HERE / name, target)
        else:
            url = f"{LABS}{folder}/{name}"
            try:
                urllib.request.urlretrieve(url, target)
            except OSError:  # Python without certificates (python.org on macOS): try curl
                if subprocess.run(["curl", "-fsSL", "-o", str(target), url]).returncode:
                    raise SystemExit(f"Could not download {url}. Check your internet connection.")
    if sha256(target) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder capacity-plan and run again.")
os.chdir(WORK)
print("Ready:", ", ".join(FILES))

## Setup: helpers

The next cell loads pandas, numpy and matplotlib, reads the recorded AI times, and defines one check helper. `expect(what, yours, expected)` compares your number with the expected one and says "too high" or "too low", without showing the answer. Run the cell. It prints nothing.

In [ ]:
# Helpers: packages, the recorded AI times, and a check that does not show the answer.
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
calibration = json.loads(Path("calibration.json").read_text())

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif abs(yours - expected) <= tolerance * max(1, abs(expected)):
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

A city's 311 line takes requests from residents by phone, app and web: a pothole, graffiti, a blocked sidewalk. Nadia runs the team that wants to add the course's AI work to every new request: classify it, draft a reply for the agent, and compute an embedding to find similar requests. She asks her engineer, Leo: "How many workers do we need, so that no request waits more than **10 seconds** before its AI work starts? And what happens when the AI provider is down?"

Leo has no traffic of his own yet. San Francisco publishes every 311 request with its time, so he plans with that trace. The [case-study page](https://learning.nextia-ai.com/courses/scaling/m07/capacity-plan-from-a-public-traffic-trace/) tells the full story; this notebook runs every step.

## 2. The data

Load the 2025 file and look at its size, its first rows and its time range.

In [ ]:
raw = pd.read_csv("sf311_2025_arrivals.csv.gz", parse_dates=["requested_datetime"],
                  keep_default_na=False, na_values=[""])
print(raw.shape)
print(raw.head(3).to_string(index=False))
print(raw.requested_datetime.min(), "to", raw.requested_datetime.max())

> **Check.** You should see `(872427, 3)`, three requests from just after midnight on 1 January 2025, and a range to 31 December 2025.

One row is one request: when the city's system recorded it, through which channel (`source`), and its type (`service_name`). Next, the first checks: the channels, empty values, repeats, and one day with a clock change.

In [ ]:
print(raw.source.value_counts(dropna=False).to_string())
print("rows with no source:", raw.source.isna().sum())
print("same time, channel and type as another row:", raw.duplicated().sum())
t = raw.requested_datetime
march_9 = raw[t.dt.date == pd.Timestamp("2025-03-09").date()]
print("2025-03-09, requests per hour 0-4:", march_9.groupby(march_9.requested_datetime.dt.hour).size().head(4).to_dict())

> **Check.** You should see `Test` with 512 rows, 1,985 rows with no source, 20,446 repeated rows, and no hour 2 on 9 March.

The page explains each one. In short: the 512 `Test` rows are not residents, so they go. The rows without a source are real requests, so they stay. The repeats are not copies: they are requests that the system recorded in the same second, the batches that this case study is about. The missing hour is the change to summer time: the times are local, without a time zone.

## 3. Frame the problem

One request is one ticket and one AI job. The question is about **waiting**: how long a job waits in the queue before a worker takes it. Leo plans with January to September (the **design months**) and keeps October to December for **one final check**, as a model is tested once on data it never saw.

In [ ]:
df = raw[raw.source != "Test"].copy()
T0 = pd.Timestamp("2025-01-01")
df["sec"] = (df.requested_datetime - T0).dt.total_seconds()  # seconds since 1 January
design = df[df.requested_datetime < "2025-10-01"]
check = df[df.requested_datetime >= "2025-10-01"]
sec = design.sec.to_numpy()
print(len(df), "requests:", len(design), "design months,", len(check), "check months")

> **Check.** You should see `871915 requests: 663926 design months, 207989 check months`.

## 4. Explore what the decision needs

### The day and the week

Count the requests per day, and the average per hour of the day, on weekdays and at weekends.

In [ ]:
days = design.groupby(design.requested_datetime.dt.date).size()
print(f"per day: mean {days.mean():.0f}, lowest {days.min()} ({days.idxmin()}), highest {days.max()} ({days.idxmax()})")
when = design.requested_datetime
weekend = when.dt.dayofweek >= 5
n_weekend = (pd.to_datetime(pd.Series(days.index)).dt.dayofweek >= 5).sum()
by_hour = pd.DataFrame({
    "weekday": design[~weekend].groupby(when[~weekend].dt.hour).size() / (len(days) - n_weekend),
    "weekend": design[weekend].groupby(when[weekend].dt.hour).size() / n_weekend,
})
print("busiest hour:", by_hour.idxmax().to_dict(), by_hour.max().round(1).to_dict())
print("quietest hour:", by_hour.idxmin().to_dict(), by_hour.min().round(1).to_dict())
by_hour.plot(xlabel="Hour of the day", ylabel="Requests per hour (average)", figsize=(8, 3))
plt.show()

> **Check.** You should see a mean of 2,432 requests per day, and a weekday peak of about 239 requests per hour at 9 in the morning.

### Three time scales

A **design point** is the load that you plan for. The average is the wrong one: it hides the peaks. But which peak? Count the requests in every hour, every minute and every 10 seconds, and compare the percentiles. The 10-second window matters because a job takes about 2.3 seconds and the target is 10 seconds.

In [ ]:
def per_window(seconds, width):
    """How many requests arrived in each window of `width` seconds (empty windows count too)."""
    start = np.floor(seconds.min() / 86400) * 86400
    return np.bincount(((seconds - start) // width).astype(int))

rows = []
for name, width in [("per hour", 3600), ("per minute", 60), ("per 10 seconds", 10)]:
    counts = per_window(sec, width)
    rows.append([name, counts.mean(), *np.percentile(counts, [50, 99, 99.9]), counts.max()])
print(pd.DataFrame(rows, columns=["window", "mean", "p50", "p99", "p99.9", "max"]).round(2).to_string(index=False))

> **Check.** You should see a mean of 1.69 per minute, a p99 minute of 8 and a busiest minute of 21; and 13 requests in the busiest 10 seconds.

### Bursts inside the minute

If residents acted independently, every second of the minute would get about 1/60 of the requests (1.7%). Check each channel.

In [ ]:
second = design.requested_datetime.dt.second
for source in ["Phone", "Mobile", "Web"]:
    counts = second[design.source == source].value_counts()
    print(f"{source:6}: second :{counts.idxmax():02d} has {counts.max() / counts.sum():.1%} of the requests")
same = design.groupby("requested_datetime").size()
print("seconds with 5 or more requests:", (same >= 5).sum(), "holding", same[same >= 5].sum(), "requests")
print("the largest group:", same.max(), "requests at", same.idxmax())
print(design[design.requested_datetime.isin(same[same >= 5].index)].source.value_counts().to_dict())

> **Check.** You should see Web with about 39% of its requests at second :47, and 1,095 seconds that hold 5 or more requests, almost all from Web. The largest group is 10 requests in one second.

A third of the web requests are recorded once a minute, in a batch. For the AI queue, the batch is the arrival: the jobs start when the system records the requests.

## 5. A baseline: the arithmetic of the average

The course's AI job is three calls in a row: classify, draft a reply, embed. Draw job times from the 100 recorded times of each call.

In [ ]:
lat = {task: np.array(ms) / 1000 for task, ms in calibration["latency_ms"].items()}
rng = np.random.default_rng(21)
job = sum(rng.choice(lat[task], 100_000) for task in ["classify", "draft_reply", "embed"]) + 0.017
print(f"one job: median {np.median(job):.2f} s, p95 {np.percentile(job, 95):.2f} s, max {job.max():.2f} s, mean {job.mean():.2f} s")
per_minute = per_window(sec, 60)
for name, rate in [("mean minute", per_minute.mean()), ("p99 minute", np.percentile(per_minute, 99)),
                   ("busiest minute", per_minute.max())]:
    print(f"{name:15} {rate:5.2f} requests -> {rate / 60 * job.mean():.2f} job slots busy on average")

> **Check.** You should see a median job of 2.25 s and, even in the busiest minute, fewer than one job slot busy (0.81).

A **job slot** is one job in flight: workers × `WORKER_CONCURRENCY` in the course project. The arithmetic says one slot is plenty. The next sections show why that answer is wrong.

## 6. The queue model

`simulate()` replays arrival times through a queue in front of a number of job slots, with the recorded job times and the provider quota (100 chat calls per minute, two per job). It returns, for each ticket, how long it waited before a slot took it. Run the cell; it prints nothing. The page explains each part.

In [ ]:
# The queue model (the same code as the case study's reference/c21/m07/l01/capacity.py).
import heapq

DB_SECONDS = 0.017  # db 9.5 ms + other 7.6 ms (the course's breakdown, Module 1)


def job_times(n, calibration, rng):
    """Seconds of each step for n jobs: arrays classify, draft_reply, embed."""
    lat = calibration["latency_ms"]
    return [rng.choice(np.array(lat[task]) / 1000, size=n) for task in ("classify", "draft_reply", "embed")]


def simulate(arrivals, slots, calibration, seed=21, rpm=100, bucket_seconds=60,
             bound=None, outage=None):
    """Replay `arrivals` (sorted seconds) and return a dict of numpy arrays:
    wait (seconds from arrival to a worker taking the job; NaN if refused),
    done (seconds when the job finished), refused (bool)."""
    arrivals = np.asarray(arrivals, dtype=float)
    n = len(arrivals)
    rng = np.random.default_rng(seed)
    classify, draft, embed = job_times(n, calibration, rng)
    rate = rpm / 60.0
    capacity = rate * bucket_seconds
    level, level_t = capacity, 0.0          # the quota bucket and when it was last updated
    out_start, out_end = outage if outage else (np.inf, np.inf)

    wait = np.full(n, np.nan)
    done = np.full(n, np.nan)
    refused = np.zeros(n, dtype=bool)
    waiting = []                             # FIFO of job numbers (a deque by index)
    head = 0
    free = slots
    events = []                              # (time, order, kind, job, step)
    order = 0
    i = 0                                    # next arrival
    woken = False                            # the "provider is back" event is planned

    def take_token(t):
        """Seconds until a chat call may start at time t (0: take the token now)."""
        nonlocal level, level_t
        level = min(capacity, level + (t - level_t) * rate)
        level_t = t
        if level >= 1 - 1e-6:                # (a tolerance for rounding)
            level -= 1
            return 0.0
        return max((1 - level) / rate, 0.001)

    def push(t, kind, job, step):
        nonlocal order
        order += 1
        heapq.heappush(events, (t, order, kind, job, step))

    def start_step(t, job, step):
        """Start step 0 (classify), 1 (draft_reply) or 2 (embed) of a job at time t, or wait."""
        if out_start <= t < out_end:
            push(out_end, "step", job, step)          # the provider is down: try when it is back
            return
        if step < 2:
            delay = take_token(t)
            if delay > 0:
                push(t + delay, "step", job, step)    # wait for the quota
                return
        seconds = (classify, draft, embed)[step][job] + (DB_SECONDS if step == 2 else 0)
        push(t + seconds, "end", job, step)

    while i < n or events:
        if i < n and (not events or arrivals[i] <= events[0][0]):
            t = arrivals[i]
            if bound is not None and len(waiting) - head >= bound:
                refused[i] = True
            else:
                waiting.append(i)
            i += 1
        else:
            t, _, kind, job, step = heapq.heappop(events)
            if kind == "step":
                start_step(t, job, step)
                continue
            if kind == "end" and step < 2:
                start_step(t, job, step + 1)
                continue
            if kind == "end":
                done[job] = t
                free += 1
        while free and head < len(waiting) and not (out_start <= t < out_end):
            job = waiting[head]
            head += 1
            free -= 1
            wait[job] = t - arrivals[job]
            start_step(t, job, 0)
        if out_start <= t < out_end and not woken:
            woken = True
            push(out_end, "wake", -1, -1)            # take jobs again when the provider is back
    return {"wait": wait, "done": done, "refused": refused}

### A tiny example

Four tickets 30 seconds apart, then ten tickets in the same second, with one slot and with two.

> **Predict.** With one slot and ten tickets at once, about how long does the tenth wait? Write it down, then run the cell.

In [ ]:
calm = [0, 30, 60, 90]
burst = [0] * 10
print("calm, 1 slot :", np.round(simulate(calm, 1, calibration)["wait"], 1))
for slots in [1, 2]:
    print(f"burst, {slots} slot(s):", np.round(simulate(burst, slots, calibration)["wait"], 1))

> **Check.** You should see no wait for the calm tickets, about 20 seconds for the tenth ticket with one slot, and about half that with two.

The same number of tickets per minute gives no queue when they are spread out, and a long one when they come together.

## 7. Three workload models, one queue, one metric

Leo compares three ways to describe the arrivals, through the same queue with the same job times. The metric is the **longest wait** in the design months, and the count of tickets that waited more than 10 seconds:

- **even**: the same number of requests, evenly spaced (the arithmetic of the average);
- **random**: random arrivals (a Poisson process) with the average of each hour of the week, the textbook model;
- **trace**: the real times, batches and all.

In [ ]:
TARGET = 10  # seconds: the longest a ticket may wait before a slot takes it

def summary(result):
    wait = result["wait"][~result["refused"]]
    return {"p99": np.percentile(wait, 99), "max": wait.max(), "over": int((wait > TARGET).sum())}

dow, hour = when.dt.dayofweek, when.dt.hour
n_dow = pd.to_datetime(pd.Series(days.index)).dt.dayofweek.value_counts().sort_index()
rate = design.groupby([dow, hour]).size().unstack(fill_value=0).div(n_dow, axis=0)

def random_arrivals(start, n_days, seed):
    """Random arrivals with the design months' average for each hour of the week."""
    rng = np.random.default_rng(seed)
    out = []
    for d in range(n_days):
        day = start + pd.Timedelta(days=d)
        for h in range(24):
            n = rng.poisson(rate.loc[day.dayofweek, h])
            out.append((day - T0).total_seconds() + 3600 * h + rng.uniform(0, 3600, n))
    return np.sort(np.concatenate(out))

models = {"even": np.linspace(sec.min(), sec.max(), len(sec)),
          "random": random_arrivals(T0, 273, seed=21), "trace": sec}
rows = [{"model": name, "slots": slots, **summary(simulate(arrivals, slots, calibration))}
        for slots in [1, 2, 3, 4] for name, arrivals in models.items()]
table = pd.DataFrame(rows)
print("longest wait (s):")
print(table.pivot(index="model", columns="slots", values="max").round(1).to_string())
print(f"tickets that waited more than {TARGET} s:")
print(table.pivot(index="model", columns="slots", values="over").to_string())

> **Check.** You should see that the even model never waits, and the random model has one ticket over 10 seconds with one slot and none with two. The trace needs three slots: with one slot 2,893 tickets waited more than 10 seconds, the longest 25.1 seconds.

## 8. The operating point

### One burst, second by second

The largest batch of the design months, replayed with 1 to 4 slots. The line is the **oldest job's age**, the number that the course watches on the queue.

In [ ]:
def oldest_age(arrivals, result, times):
    """The age of the oldest waiting job at each time (0 when nothing waits)."""
    start = arrivals + result["wait"]
    return [max((x - a for a, s in zip(arrivals, start) if a <= x < s), default=0) for x in times]

at = (same.idxmax() - T0).total_seconds()
window = sec[(sec >= at - 120) & (sec < at + 120)]
times = np.arange(at - 10, at + 60, 0.5)
for slots in [1, 2, 3, 4]:
    result = simulate(window, slots, calibration)
    plt.plot(times - at, oldest_age(window, result, times), label=f"{slots} slot(s)")
    print(f"{slots} slot(s): longest wait {np.nanmax(result['wait']):.1f} s")
plt.axhline(TARGET, color="grey", linestyle="--")
plt.xlabel("Seconds after the batch"); plt.ylabel("Oldest job's age (s)"); plt.legend(); plt.show()

> **Check.** You should see about 21 s with one slot, about 10 s with two, and under 7 s with three or four.

### Five passes, not one

The job times are random, so one run is one sample. Run each choice with five seeds and keep the median, as the course does with load tests.

In [ ]:
def passes(arrivals, slots, seeds=(21, 22, 23, 24, 25), **kw):
    runs = pd.DataFrame([summary(simulate(arrivals, slots, calibration, seed=s, **kw)) for s in seeds])
    return {"median max": round(float(runs["max"].median()), 1),
            "max range": (round(float(runs["max"].min()), 1), round(float(runs["max"].max()), 1)),
            "median over": int(runs["over"].median())}

for slots in [2, 3, 4]:
    print(f"design months, {slots} slots:", passes(sec, slots))

> **Check.** You should see a median longest wait of about 11.3 s with two slots (12 tickets over 10 s), 7.0 s with three and 4.9 s with four.

Three slots meet the target in the design months. Leo recommends **four**: the cost of a slot is small (a worker mostly waits for the provider), and the design months may not hold the worst burst. Now the one check that can show it.

## 9. The final check, once

Run October to December through the same model, once, with two, three and four slots.

In [ ]:
check_sec = check.sec.to_numpy()
check_ten = per_window(check_sec, 10)
print("check months: busiest 10 seconds", check_ten.max(), "requests (design months: 13)")
for slots in [2, 3, 4]:
    print(f"check months, {slots} slots:", passes(check_sec, slots))

> **Check.** You should see 22 requests in the busiest 10 seconds, a median longest wait of about 16.8 s with two slots, 9.0 s with three and 5.5 s with four.

The check months had a burst bigger than any in the design months, and three slots only just held. The margin of the fourth slot was needed. The page shows what that burst was.

## 10. The quota, an outage and a queue bound

The busiest minute needs 2 × 21 = 42 chat calls; the quota allows 100. Now an outage: the provider is down for 30 minutes from the start of the busiest hour. Work waits; when the provider is back, the quota limits how fast the queue empties. Compare more slots, a larger quota, and two queue bounds.

In [ ]:
def queued(arrivals, result, times):
    """Jobs waiting at each time (accepted, not yet taken)."""
    start = arrivals + result["wait"]
    return np.array([((arrivals <= x) & ~result["refused"] & ~(start <= x)).sum() for x in times])

hour_counts = per_window(sec, 3600)
h0 = np.floor(sec.min() / 86400) * 86400 + 3600 * hour_counts.argmax()
print("busiest hour:", T0 + pd.Timedelta(seconds=h0), "with", hour_counts.max(), "requests")
outage = (h0, h0 + 1800)
win = sec[(sec >= h0 - 600) & (sec < h0 + 3 * 3600)]
grid = np.arange(h0 - 300, h0 + 2 * 3600, 30)
for name, slots, kw in [("4 slots", 4, {}), ("8 slots", 8, {}), ("4 slots, quota 200", 4, {"rpm": 200}),
                        ("4 slots, bound 500", 4, {"bound": 500}), ("4 slots, bound 100", 4, {"bound": 100})]:
    result = simulate(win, slots, calibration, outage=outage, **kw)
    q = queued(win, result, grid)
    empty = grid[(grid > outage[1]) & (q == 0)][0]
    print(f"{name:19} peak {q.max():3} waiting, refused {result['refused'].sum():3}, "
          f"longest wait {np.nanmax(result['wait']) / 60:.1f} min, empty {(empty - outage[1]) / 60:.1f} min after the outage")

> **Check.** You should see 273 jobs waiting at the peak and the queue empty 5.5 minutes after the outage, with 4 or 8 slots. A quota of 200 empties it in 3.0 minutes. A bound of 100 refuses 177 requests.

More slots do not help after an outage: the quota does. And a bound of 100 refuses 177 residents' requests, which the course's design does not save. The page discusses what a city should do instead.

## 11. The plan on new input, end to end

`plan()` takes a new trace file and runs the whole method: checks, the three time scales, the quota, five passes with the chosen slots, and a verdict. It refuses to extrapolate: if the new trace has a burst bigger than any that the plan was checked on, it says so instead of giving a number.

In [ ]:
CHECKED = {"per 10 seconds": max(per_window(sec, 10).max(), check_ten.max()),
           "per minute": max(per_minute.max(), per_window(check_sec, 60).max())}

def plan(path, slots=4):
    new = pd.read_csv(path, parse_dates=["requested_datetime"], keep_default_na=False, na_values=[""])
    new = new[new.source != "Test"]
    s = (new.requested_datetime - new.requested_datetime.min().normalize()).dt.total_seconds().to_numpy()
    m, ten = per_window(s, 60), per_window(s, 10)
    print(f"{len(new)} requests, {new.requested_datetime.min():%Y-%m-%d} to {new.requested_datetime.max():%Y-%m-%d}")
    print(f"per minute: mean {m.mean():.2f}, p99 {np.percentile(m, 99):.0f}, max {m.max()} (checked up to {CHECKED['per minute']})")
    print(f"per 10 seconds: max {ten.max()} (checked up to {CHECKED['per 10 seconds']})")
    print(f"busiest minute: {2 * m.max()} chat calls (quota 100)")
    if ten.max() > CHECKED["per 10 seconds"] or m.max() > CHECKED["per minute"] or 2 * m.max() > 100:
        print("Verdict: outside what the plan was checked on. Measure again; do not extrapolate.")
        return None
    result = passes(s, slots)
    print(f"{slots} slots:", result)
    verdict = "meets" if result["max range"][1] <= TARGET else "misses"
    print(f"Verdict: {slots} slots {verdict} the {TARGET} s target on this input.")
    return result

june = plan("sf311_2026_06_arrivals.csv.gz")

> **Check.** You should see 73,719 requests in June 2026, a busiest minute of 21 and 12 requests in the busiest 10 seconds, both inside the checked range, and a verdict that four slots meet the target (longest wait about 4 s).

## 12. Your turn

### Task 1: a 30-second window

> **Your turn.** Count the requests of the design months per 30 seconds with `per_window()`, and put the **p99.9** of those counts in `p999_30s`.

In [ ]:
p999_30s = ...  # your code here

In [ ]:
expect("the p99.9 of requests per 30 seconds", p999_30s, 9.0)

### Task 2: a stricter target

> **Your turn.** Nadia asks for 5 seconds instead of 10. Find the smallest number of slots for which the longest wait in the design months, with the default seed, is under 5 seconds. Put it in `slots_for_5s`.

In [ ]:
slots_for_5s = ...  # your code here

In [ ]:
expect("the smallest number of slots for 5 seconds", slots_for_5s, 4)

### Task 3: a longer outage

> **Your turn.** Repeat the outage of section 10 with 4 slots and a bound of 500, but the provider is down for **90 minutes**. How many requests does the bound refuse? Put the number in `refused_90`.

In [ ]:
refused_90 = ...  # your code here

In [ ]:
expect("the requests refused in a 90-minute outage", refused_90, 221)

> **Check.** If a check says "too high" or "too low", read the task again. The [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C21/M07-L01-capacity-plan-from-a-public-traffic-trace/capacity-plan-from-a-public-traffic-trace-solution.ipynb) has the answers.

## Recap

- The average said one slot is plenty. The real arrivals come in batches inside the minute, and three slots were needed for a 10-second target in the design months.
- The design point is set at the time scale of the target and the job: 10-second windows, not minutes or hours.
- The held-out months had a bigger burst. Four slots, chosen with a margin, held; three only just did.
- After an outage, the quota, not the number of workers, sets how fast the queue empties. A tight queue bound would refuse residents' requests.
- `plan()` checks new input against the range that the plan was tested on, and refuses to extrapolate.

The knowledge checks are on the [case-study page](https://learning.nextia-ai.com/courses/scaling/m07/capacity-plan-from-a-public-traffic-trace/).

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Queue and cache for an open-data API](https://learning.nextia-ai.com/courses/scaling/m07/queue-and-cache-for-an-open-data-api/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/scaling/m07/capacity-plan-from-a-public-traffic-trace/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The 311 data is real public data (ODC PDDL 1.0); it has no names or addresses in this extract.